# PURITY Inference

Runs the unified config-driven inference pipeline for PURITY.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pyarrow.parquet as pq

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = zenml_utils.setup_repo_pythonpath(Path(zenml_utils.find_project_root()).resolve())
from pioneerml_purity_plugin.purity.pipeline import inference_pipeline, load_config

zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)
sys.path.insert(0, str(PROJECT_ROOT / 'artifacts'))
from generate_purity_dummy_parquet import generate_dummy_purity_parquet


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## Build Input + Resolve Model

In [2]:
input_parquet = PROJECT_ROOT / 'artifacts' / 'purity_notebook_inference.parquet'
generate_dummy_purity_parquet(output_path=input_parquet, num_events=32, seed=23)

model_path_file = PROJECT_ROOT / 'artifacts' / 'purity_small_model_path.txt'
if model_path_file.exists():
    model_path = Path(model_path_file.read_text(encoding='utf-8').strip()).resolve()
else:
    candidates = sorted((PROJECT_ROOT / 'artifacts' / 'purity_notebook_export').glob('*_state_dict.pt'))
    if not candidates:
        fallback = sorted((PROJECT_ROOT / 'artifacts' / 'purity_notebook_export').glob('*_torchscript.pt'))
        if not fallback:
            raise FileNotFoundError('Run training notebook first to produce a PURITY export.')
        model_path = fallback[-1].resolve()
    else:
        model_path = candidates[-1].resolve()

model_path


PosixPath('/workspace/artifacts/purity_small_export/purity_small_20260415_232149_20260415_232212_state_dict.pt')

## Patch Config and Run

In [3]:
cfg = load_config()['inference']
cfg['model_handle_builder']['model_handle']['type'] = 'purity_eager'
cfg['model_handle_builder']['model_handle']['config']['model_path'] = str(model_path)
cfg['inference']['loader_manager']['config']['input_sources_spec']['main_sources'] = [str(input_parquet)]
cfg['inference']['loader_manager']['config']['input_sources_spec']['optional_sources_by_name'] = {}
cfg['inference']['loader_manager']['config']['input_sources_spec']['source_type'] = 'file'
cfg['inference']['writer']['config']['output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['fallback_output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['write_timestamped'] = False

run = inference_pipeline.with_options(enable_cache=False)(pipeline_config=cfg)
out = load_step_output(run, 'run_inference')
out


Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
[build_model_handle] No materializer is registered for type <class 'pioneerml_purity_plugin.purity.model_handle.purity_eager_model_handle.PurityEagerModelHandle'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml_purity_plugin.purity.model_handle.purity_eager_model_handle.PurityEagerModelHandle'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step b

{'predictions_path': '/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet',
 'predictions_paths': ['/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet'],
 'timestamped_predictions_path': None,
 'timestamped_predictions_paths': []}

In [4]:
pred_path = Path(load_step_output(run, 'run_inference')['predictions_path'])
tbl = pq.read_table(pred_path)
print(pred_path)
print(tbl.schema)
tbl.slice(0, 3).to_pydict()

/workspace/artifacts/purity_notebook_predictions/purity_notebook_inference_preds.parquet
event_id: int64
pred_purity_signal: list<element: float>
  child 0, element: float
pred_purity_logit: list<element: float>
  child 0, element: float
pred_purity_summary_accepted: list<element: float>
  child 0, element: float
pred_purity_summary_positron_energy: list<element: float>
  child 0, element: float
pred_purity_summary_positron_time: list<element: float>
  child 0, element: float
pred_purity_summary_positron_polar_angle: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64


{'event_id': [0, 1, 2],
 'pred_purity_signal': [[0.4965341091156006,
   0.3911376893520355,
   0.492922306060791,
   0.39031097292900085,
   0.49817976355552673,
   0.3931777775287628,
   0.4953083097934723],
  [0.3916267156600952,
   0.48557212948799133,
   0.49967846274375916,
   0.39261335134506226,
   0.4843922555446625,
   0.49923840165138245,
   0.39309772849082947],
  [0.4830796718597412,
   0.49758219718933105,
   0.39286118745803833,
   0.4938017427921295,
   0.46429359912872314,
   0.4994913637638092,
   0.39037173986434937]],
 'pred_purity_logit': [[-0.013863801956176758,
   -0.4425324499607086,
   -0.02831260859966278,
   -0.446005254983902,
   -0.007280975580215454,
   -0.4339739680290222,
   -0.01876729726791382],
  [-0.4404796361923218,
   -0.0577273964881897,
   -0.0012860000133514404,
   -0.4363403916358948,
   -0.06245142221450806,
   -0.0030464529991149902,
   -0.43430963158607483],
  [-0.0677071213722229,
   -0.009671226143836975,
   -0.4353009760379791,
   -0.02479